In [1]:
from pathlib import Path
import duckdb

In [ ]:
# duckdb.connect(':memory:')

In [2]:
# Initialize data Path & database file
data_path = Path('../data/')
db_file = '../olist_medallion_architecture.db'

# Connect to database file
conn = duckdb.connect(db_file)

### Build bronze layer objects
* Simple views w/o any changes to the underlying raw data

In [3]:
# Terms for removal
remove_set = set(['olist', 'order', 'dataset'])

# List raw Olist CSV file names
for p in data_path.glob('*.csv'):
    # Remove terms from file name
    file_name_list = p.stem.split('_')

    # Simplify table name from file name
    view_name_list = [x for x in file_name_list if x not in remove_set]
    view_name = '_'.join(view_name_list)

    # Create bronze views
    print(f'Creating bronze_{view_name}')
    conn.execute(f"""
        CREATE OR REPLACE VIEW bronze_{view_name} AS 
        SELECT * FROM read_csv_auto('{data_path}/{p.name}');
    """)

print(f'\nBronze layer views created...')

Creating bronze_brazil_state_code_ref
Creating bronze_customers
Creating bronze_geolocation
Creating bronze_orders
Creating bronze_items
Creating bronze_payments
Creating bronze_reviews
Creating bronze_products
Creating bronze_sellers
Creating bronze_product_category_name_translation

Bronze layer views created...


### Validation Queries

In [4]:
# Verify views exist in data dictionary
conn.query("""
    SELECT view_name
    FROM duckdb_views() 
    WHERE NOT internal
    ORDER BY
        view_name;
""")

view_nameVARCHAR
bronze_brazil_state_code_ref
bronze_customers
bronze_geolocation
bronze_items
bronze_orders
bronze_payments
bronze_product_category_name_translation
bronze_products
bronze_reviews
bronze_sellers


In [5]:
# Evaluate view records
conn.query("""
    SELECT *
    FROM bronze_payments
    WHERE order_id = 'ea9184ad433a404df1d72fa0a8764232'
    ORDER BY payment_sequential;
""")

order_idVARCHAR,payment_sequentialBIGINT,payment_typeVARCHAR,payment_installmentsBIGINT,payment_valueDOUBLE
ea9184ad433a404df1d72fa0a8764232,1,credit_card,1,17.78
ea9184ad433a404df1d72fa0a8764232,2,voucher,1,22.49
ea9184ad433a404df1d72fa0a8764232,3,voucher,1,22.49
ea9184ad433a404df1d72fa0a8764232,4,voucher,1,22.49
ea9184ad433a404df1d72fa0a8764232,5,voucher,1,22.47


In [6]:
# Close database connection
conn.close()